# 01 — Metal binding (GEAR metal-only v2 + GEAR Ultimate archive)
Feature discovery first (same MLP), architecture second, external confirmation last.
Features + trainer come from the GEAR backend on Drive; every grid resumes from its per-fold cache.

In [ ]:
# ============================================================
# BOOTSTRAP — Colab: mount Drive, put pbind on the path
# ============================================================
import sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# Folder of this repo (copy it to Drive once, or git clone it into /content).
REPO = Path("/content/drive/MyDrive/Protein-binding-Phi/protein-binding")
sys.path.insert(0, str(REPO / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)

# Data root (Drive layout). Override with PBIND_BASE for a local mirror.
# import os; os.environ["PBIND_BASE"] = "/content/drive/MyDrive/Protein-binding-Phi"

from pbind.paths import resolve_paths, check_inputs
display(check_inputs(resolve_paths()))

In [ ]:
from pbind.pipelines import open_workspace, metal

ws = open_workspace("metal")            # GEAR_ULTIMATE backend, MLP/KAN/XGBoost context
A = metal.audit(ws); display(A["context"]); display(A["dims"])

## Stage A–C: family screen, component screen, combination ladder + Dev-only lock

In [ ]:
display(metal.family_screen(ws))
display(metal.component_screen(ws))
lock = metal.combination_ladder(ws, policy="best_mean")   # or "one_se"
lock

## Stage D: model family on the locked representation (Dev only)

In [ ]:
metal.model_family_screen(ws)

## Stage E (optional): graph diagnostic

In [ ]:
RUN_GRAPH = False
if RUN_GRAPH:
    graph_cv, graph_ext = metal.graph_diagnostic(ws, include_esm=False)

## Stage F–H: final locked external run, frozen baselines, manifest

In [ ]:
cv, ext = metal.final_external(ws)
display(cv); display(ext.sort_values(["dataset", "AUPRC"], ascending=[True, False]))
display(metal.baselines(ws))
metal.manifest(ws)

## Archive: original GEAR Ultimate grids (switch on what you need)
`ultimate_grid(ws, which)` with which ∈ standardized_13 | mechanistic | final_focused | no_esm | evo_full;
`ultimate_graph(ws, which)` with maximal | clean | evo; EVO: `msa_batches` → `evo_dev_cv`.

In [ ]:
RUN = dict(standardized_13=False, mechanistic=False, final_focused=False, no_esm=False)
for which, on in RUN.items():
    if on:
        cv, ext, paired = metal.ultimate_grid(ws, which, models=("MLP",))
        display(cv); display(ext)

# metal.msa_batches(ws, splits=("train",), batch_size=25, max_batches=5)   # Dev first
# display(metal.evo_dev_cv(ws))
# metal.historical_kan(ws)